# Extract Waveform Parameters from ESM-DB ASDF Files

**Author:** Spina Cianetti  
**License:** [GPL-3.0](https://www.gnu.org/licenses/gpl-3.0.html)  
This code is released under the GNU General Public License v3.0.

---

This notebook extracts waveform metadata from ESM-DB ASDF (`.h5`) files and produces
three output files:

| File | Description |
|------|-------------|
| `waveform_summary.csv` | One row per trace: station coordinates, event origin parameters, processing type, duration, sampling rate |
| `cv_mp_duration_comparison.csv` | Per-trace CV vs MP duration comparison |
| `files_without_mp.txt` | Events with no MP-processed traces |

**Key design choices:**
- Networks stored as `IV` in the ASDF but carrying a different real code (`8H`, `5G`, `8P`, `9N`)
  are corrected by reading the actual network from the ASDF waveform group name instead of
  `trace.stats.network`, which ObsPy would misread as `IV` for those networks.
- Station coordinates are read from the embedded StationXML inventory.
- Event origin parameters (location, depth, uncertainties, azimuthal gap) are read from
  the embedded QuakeML catalog.

**Sections:**
1. Setup and configuration
2. Waveform parameter extraction
3. Dataset overview and quality statistics
4. Diagnostic plots
5. Cross-validation utilities

## 1. Setup and configuration

In [ ]:
import os

# Disable HDF5 file locking — required on network/NFS filesystems
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

import pyasdf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.notebook import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_DIR  = '/home/jovyan/shared/users/spina/ESM25/output'
FIGURE_DIR = 'FIGURE'
os.makedirs(FIGURE_DIR, exist_ok=True)

# Output CSV / text files
OUT_SUMMARY    = 'waveform_summary.csv'
OUT_CV_MP_DIFF = 'cv_mp_duration_comparison.csv'
OUT_NO_MP      = 'files_without_mp.txt'
OUT_MISS_DEPTH = 'filenames_missing_depth.txt'

# ── Network code correction ───────────────────────────────────────────────────
# These networks are stored under the 'IV' code inside trace.stats.network by
# ObsPy, but their real network code is different. The correct code is recovered
# from the ASDF waveform group name (e.g. '8H.RM02').
SPECIAL_NETS = {'8H', '5G', '8P', '9N'}

# ── Geographic bounding box (Europe / Mediterranean study region) ─────────────
LAT_MIN, LAT_MAX =   0,  90
LON_MIN, LON_MAX = -30,  60

# ── Collect all HDF5 files ────────────────────────────────────────────────────
all_h5_files = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith('.h5'))
print(f'HDF5 files found: {len(all_h5_files)}')

## 2. Waveform parameter extraction

Iterates over all ASDF files and extracts for each trace:

- **Station metadata**: network, station code, location, channel, latitude, longitude, elevation
- **Event origin**: time, latitude, longitude, depth, azimuthal gap, and associated uncertainties
- **Trace metadata**: processing type (CV / MP), tag, start/end time, duration, sampling rate, npts

A per-file CV vs MP duration comparison table is also built on the fly.

Results are written to `waveform_summary.csv` and `cv_mp_duration_comparison.csv`.

In [ ]:
records          = []   # one dict per trace
files_without_mp = []   # events with no MP-processed traces
cv_mp_diff       = []   # CV vs MP duration comparison, one dict per trace pair

for fname in tqdm(all_h5_files, desc='Extracting waveform parameters', unit='file'):
    file_path    = os.path.join(INPUT_DIR, fname)
    file_has_mp  = False
    file_records = []

    try:
        with pyasdf.ASDFDataSet(file_path, mode='r', mpi=False) as ds:

            # ── Event origin parameters (QuakeML) ────────────────────────────
            origin_time           = None
            origin_time_error     = None
            epicenter_lat         = None
            epicenter_lon         = None
            epicenter_depth_m     = None
            epicenter_gap         = None
            epicenter_lat_error   = None
            epicenter_lon_error   = None
            epicenter_depth_error = None
            try:
                if ds.events:
                    origin = ds.events[0].origins[0]
                    origin_time           = str(origin.time) if origin.time else None
                    origin_time_error     = (origin.time_errors.uncertainty
                                             if origin.time_errors else None)
                    epicenter_lat         = origin.latitude
                    epicenter_lon         = origin.longitude
                    epicenter_depth_m     = origin.depth        # metres
                    epicenter_gap         = (origin.quality.azimuthal_gap
                                             if origin.quality else None)
                    epicenter_lat_error   = (origin.latitude_errors.uncertainty
                                             if origin.latitude_errors else None)
                    epicenter_lon_error   = (origin.longitude_errors.uncertainty
                                             if origin.longitude_errors else None)
                    epicenter_depth_error = (origin.depth_errors.uncertainty
                                             if origin.depth_errors else None)
            except Exception:
                pass  # event info unavailable — leave all fields as None

            for station_group_name in ds.waveforms.list():
                # The ASDF group name (e.g. '8H.RM02') holds the real network code.
                # For SPECIAL_NETS ObsPy would return 'IV' from trace.stats.network,
                # so we always use the group name as the authoritative source.
                real_net  = station_group_name.split('.')[0]
                waveforms = ds.waveforms[station_group_name]
                tags      = waveforms.get_waveform_tags()

                # ── Station coordinates (StationXML) ─────────────────────────
                sta_lat  = None
                sta_lon  = None
                sta_elev = None
                try:
                    inv_station = waveforms.StationXML[0][0]
                    sta_lat  = inv_station.latitude
                    sta_lon  = inv_station.longitude
                    sta_elev = inv_station.elevation
                except Exception:
                    pass

                for tag in tags:
                    try:
                        stream = waveforms[tag]
                        for trace in stream:
                            start     = trace.stats.starttime
                            end       = trace.stats.endtime
                            duration  = end - start
                            proc_type = ('cv'      if 'cv'  in tag
                                         else 'mp' if 'mp'  in tag
                                         else 'unknown')
                            if proc_type == 'mp':
                                file_has_mp = True

                            rec = {
                                'filename':                fname,
                                'station':                 f'{real_net}.{trace.stats.station}',
                                'location':                trace.stats.location,
                                'channel':                 trace.stats.channel,
                                'station_latitude':        sta_lat,
                                'station_longitude':       sta_lon,
                                'station_elevation_m':     sta_elev,
                                'event_origin_time':       origin_time,
                                'event_origin_time_error': origin_time_error,
                                'event_latitude':          epicenter_lat,
                                'event_longitude':         epicenter_lon,
                                'event_depth_m':           epicenter_depth_m,
                                'event_azimuthal_gap':     epicenter_gap,
                                'event_latitude_error':    epicenter_lat_error,
                                'event_longitude_error':   epicenter_lon_error,
                                'event_depth_error':       epicenter_depth_error,
                                'processing':              proc_type,
                                'tag':                     tag,
                                'start_time':              start.isoformat(),
                                'end_time':                end.isoformat(),
                                'duration_sec':            duration,
                                'sampling_rate_hz':        trace.stats.sampling_rate,
                                'time_step_sec':           trace.stats.delta,
                                'npts':                    trace.stats.npts,
                            }
                            records.append(rec)
                            file_records.append(rec)
                    except Exception as e:
                        print(f'  Error reading tag "{tag}" in {fname}: {e}')

    except Exception as e:
        print(f'  Error opening {fname}: {e}')
        continue

    if not file_has_mp:
        files_without_mp.append(fname)

    # ── CV vs MP duration comparison (per file) ───────────────────────────────
    df_file = pd.DataFrame(file_records)
    if not df_file.empty:
        pivot = df_file.pivot_table(
            index=['station', 'location', 'channel'],
            columns='processing',
            values='duration_sec',
            aggfunc='first',
        ).reset_index()
        for _, row in pivot.iterrows():
            if pd.notna(row.get('cv')) and pd.notna(row.get('mp')):
                d = abs(row['cv'] - row['mp'])
                cv_mp_diff.append({
                    'filename':      fname,
                    'station':       row['station'],
                    'location':      row['location'],
                    'channel':       row['channel'],
                    'duration_cv':   row['cv'],
                    'duration_mp':   row['mp'],
                    'duration_diff': d,
                    'match':         d <= 1.0,
                })

# ── Save outputs ──────────────────────────────────────────────────────────────
df_all  = pd.DataFrame(records)
df_diff = pd.DataFrame(cv_mp_diff)

df_all.to_csv(OUT_SUMMARY,    index=False)
df_diff.to_csv(OUT_CV_MP_DIFF, index=False)
with open(OUT_NO_MP, 'w') as f:
    f.writelines(fn + '\n' for fn in files_without_mp)

n_special = df_all[df_all['station'].str.split('.').str[0].isin(SPECIAL_NETS)].shape[0]
print('\n── Extraction complete ─────────────────────────────────────────────')
print(f'  Total traces saved            : {len(df_all):>10,}')
print(f'  SPECIAL_NET traces corrected  : {n_special:>10,}  (8H / 5G / 8P / 9N)')
print(f'  Events without MP traces      : {len(files_without_mp):>10}')
print(f'  Saved → {OUT_SUMMARY}')
print(f'  Saved → {OUT_CV_MP_DIFF}')
print(f'  Saved → {OUT_NO_MP}')

## 3. Dataset overview and quality statistics

Loads the summary CSV, applies the geographic bounding box filter and prints key
statistics broken down by processing type (CV / MP).

In [ ]:
# Load (can be run independently after extraction)
df_all = pd.read_csv(OUT_SUMMARY, dtype={'station': str, 'location': str})
print(f'Total rows loaded: {len(df_all):,}')
df_all.head(3)

In [ ]:
# ── Geographic filter ────────────────────────────────────────────────────────
df_sel = df_all[
    df_all['event_latitude'].between(LAT_MIN, LAT_MAX) &
    df_all['event_longitude'].between(LON_MIN, LON_MAX)
].copy()
n_removed = len(df_all) - len(df_sel)
print(f'Rows after geographic filter : {len(df_sel):,}  (removed {n_removed:,})')

# Separate by processing type
df_cv = df_sel[df_sel['processing'] == 'cv']
df_mp = df_sel[df_sel['processing'] == 'mp']

# Event-level view: one row per (filename, processing)
df_events_cv = df_cv.drop_duplicates(subset=['filename'])
df_events_mp = df_mp.drop_duplicates(subset=['filename'])

print(f'\n  Processing   Unique events   Total traces')
print(f'  {"─"*38}')
print(f'  CV           {len(df_events_cv):>12,}   {len(df_cv):>12,}')
print(f'  MP           {len(df_events_mp):>12,}   {len(df_mp):>12,}')

# Unique stations
n_sta = df_sel['station'].nunique()
print(f'\n  Unique stations in filtered dataset: {n_sta:,}')

# Events with missing depth
n_miss_depth = df_events_cv['event_depth_m'].isna().sum()
print(f'  CV events missing event_depth_m    : {n_miss_depth}')

# Stations per event (MP)
sta_per_event = df_mp.groupby('filename')['station'].nunique()
print(f'\n  Stations per event (MP):')
print(f'    min={sta_per_event.min()}  median={sta_per_event.median():.0f}  '
      f'max={sta_per_event.max()}  mean={sta_per_event.mean():.1f}')

In [ ]:
# Save list of events with missing depth for re-download or manual inspection
mask_no_depth = df_events_cv['event_depth_m'].isna()
df_events_cv.loc[mask_no_depth, 'filename'].to_csv(
    OUT_MISS_DEPTH, index=False, header=False)
print(f'{mask_no_depth.sum()} event(s) with missing depth saved to {OUT_MISS_DEPTH}')

## 4. Diagnostic plots

The following figures are produced and saved to the `FIGURE/` directory:

| Figure | Description |
|--------|-------------|
| `hist_cv.png` / `hist_mp.png` | Distributions of all numeric parameters |
| `epicentre_map.png` | Epicentre map coloured by depth |
| `station_map.png` | Recording station locations |
| `duration_distribution.png` | Waveform duration histograms (CV vs MP) |
| `cv_mp_duration_scatter.png` | CV vs MP duration scatter with 1:1 line |
| `traces_per_event.png` | Distribution of trace count per event |
| `sampling_rate.png` | Sampling rate distribution |

In [ ]:
def annotated_hist(df, title, figpath):
    """Histograms of all numeric columns with valid-count annotations."""
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        print(f'No numeric columns in {title}')
        return
    n_cols = 4
    n_rows = int(np.ceil(len(numeric_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5 * n_cols, 3.5 * n_rows))
    axes = axes.flatten()
    total = len(df)
    for ax, col in zip(axes, numeric_cols):
        data = df[col].dropna()
        ax.hist(data, bins=30, color='steelblue',
                edgecolor='white', linewidth=0.4)
        ax.set_title(col, fontsize=9)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
        n_valid = len(data)
        ax.text(0.97, 0.95, f'N={n_valid:,} / {total:,}',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=8, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.75, edgecolor='none'))
    for ax in axes[len(numeric_cols):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(figpath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {figpath}')


# Event-level dataframes for histograms (one row per event)
annotated_hist(
    df_events_cv,
    'Parameter distributions — CV processing (one row per event)',
    os.path.join(FIGURE_DIR, 'hist_cv.png'),
)
annotated_hist(
    df_events_mp,
    'Parameter distributions — MP processing (one row per event)',
    os.path.join(FIGURE_DIR, 'hist_mp.png'),
)

In [ ]:
# ── Epicentre map (coloured by depth) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
depth_km = df_events_cv['event_depth_m'].dropna() / 1000
sc = ax.scatter(
    df_events_cv.loc[depth_km.index, 'event_longitude'],
    df_events_cv.loc[depth_km.index, 'event_latitude'],
    c=depth_km, cmap='plasma_r', s=10, alpha=0.65,
    linewidths=0, vmin=0, vmax=300,
)
plt.colorbar(sc, ax=ax, label='Depth (km)', pad=0.01, shrink=0.85)
ax.set_xlim(LON_MIN, LON_MAX)
ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
ax.set_title(
    f'Epicentre map — {len(df_events_cv):,} events (CV dataset)',
    fontweight='bold')
ax.grid(True, linewidth=0.3, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'epicentre_map.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Station map ───────────────────────────────────────────────────────────────
df_sta = (
    df_sel.drop_duplicates(subset=['station'])
          .dropna(subset=['station_latitude', 'station_longitude'])
)
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(
    df_sta['station_longitude'], df_sta['station_latitude'],
    c='darkorange', s=14, alpha=0.7, linewidths=0,
    label=f'Stations  N={len(df_sta):,}',
)
ax.set_xlim(LON_MIN, LON_MAX)
ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
ax.set_title('Recording station locations', fontweight='bold')
ax.legend(markerscale=2, fontsize=9)
ax.grid(True, linewidth=0.3, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'station_map.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Waveform duration distributions (CV vs MP, all traces) ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
for ax, df_proc, label, color in zip(
    axes,
    [df_cv, df_mp],
    ['CV', 'MP'],
    ['steelblue', 'tomato'],
):
    data = df_proc['duration_sec'].dropna()
    ax.hist(data, bins=50, color=color, edgecolor='white', linewidth=0.3)
    med = data.median()
    ax.axvline(med, color='black', linestyle='--', linewidth=1.2,
               label=f'Median: {med:.1f} s')
    ax.set_xlabel('Duration (s)')
    ax.set_ylabel('Number of traces')
    ax.set_title(f'Trace duration — {label}  (N={len(data):,})',
                 fontweight='bold')
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'duration_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CV vs MP duration scatter plot ───────────────────────────────────────────
df_cmp = pd.read_csv(OUT_CV_MP_DIFF)
match    = df_cmp[df_cmp['match']]
no_match = df_cmp[~df_cmp['match']]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(match['duration_cv'], match['duration_mp'],
           s=2, alpha=0.25, color='steelblue',
           label=f'|Δ| ≤ 1 s  (N={len(match):,})')
ax.scatter(no_match['duration_cv'], no_match['duration_mp'],
           s=5, alpha=0.6, color='tomato',
           label=f'|Δ| > 1 s  (N={len(no_match):,})')

# 1:1 reference line
lim = max(df_cmp['duration_cv'].max(), df_cmp['duration_mp'].max()) * 1.03
ax.plot([0, lim], [0, lim], 'k--', linewidth=0.9, label='1 : 1 line')

ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel('CV duration (s)')
ax.set_ylabel('MP duration (s)')
ax.set_title('CV vs MP trace duration comparison', fontweight='bold')
ax.legend(fontsize=9, markerscale=3)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'cv_mp_duration_scatter.png'),
            dpi=150, bbox_inches='tight')
plt.show()

pct = 100 * len(match) / len(df_cmp) if len(df_cmp) else 0
print(f'Traces with |CV − MP| ≤ 1 s: {len(match):,} / {len(df_cmp):,}  ({pct:.1f}%)')

In [ ]:
# ── Traces per event (CV and MP) ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, df_proc, label, color in zip(
    axes,
    [df_cv, df_mp],
    ['CV', 'MP'],
    ['steelblue', 'tomato'],
):
    counts = df_proc.groupby('filename').size()
    med = counts.median()
    ax.hist(counts, bins=40, color=color, edgecolor='white', linewidth=0.3)
    ax.axvline(med, color='black', linestyle='--', linewidth=1.2,
               label=f'Median: {med:.0f}')
    ax.set_xlabel('Number of traces per event')
    ax.set_ylabel('Number of events')
    ax.set_title(f'Traces per event — {label}', fontweight='bold')
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'traces_per_event.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sampling rate distribution ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sr_counts = df_sel['sampling_rate_hz'].value_counts().sort_index()
ax.bar(
    sr_counts.index.astype(str), sr_counts.values,
    color='steelblue', edgecolor='white', linewidth=0.4,
)
ax.set_xlabel('Sampling rate (Hz)')
ax.set_ylabel('Number of traces')
ax.set_title('Sampling rate distribution — all traces', fontweight='bold')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'sampling_rate.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Station elevation distribution ───────────────────────────────────────────
# Stations with elevation > 9000 m are likely placeholder / error values.
df_elev = df_sta[df_sta['station_elevation_m'] <= 9000]
n_suspect = len(df_sta) - len(df_elev)
if n_suspect:
    print(f'Stations with elevation > 9000 m (excluded from plot): {n_suspect}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_elev['station_elevation_m'], bins=40,
        color='seagreen', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Elevation (m a.s.l.)')
ax.set_ylabel('Number of stations')
ax.set_title(f'Station elevation distribution  (N={len(df_elev):,})',
             fontweight='bold')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'station_elevation.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 5. Cross-validation utilities

Two helper functions for comparing the current extraction against a previously
generated reference CSV, useful for detecting regressions between runs:

- **`count_missing`** — multiset comparison: finds key combinations present
  more times in the new extraction than in the reference (aggregated report).
- **`anti_join`** — left anti-join: returns rows of the new extraction that
  have no match at all in the reference on the given key columns.

In [ ]:
def count_missing(
    new_df: pd.DataFrame,
    ref_df: pd.DataFrame,
    on: list | None = None,
    normalize: bool = True,
    na_equal: bool = True,
    expand: bool = False,
    fix_typo: bool = True,
) -> pd.DataFrame:
    """Return key combinations present in new_df more times than in ref_df.

    Parameters
    ----------
    new_df, ref_df : DataFrames to compare.
    on             : Key columns; defaults to the common column set.
    normalize      : Cast keys to string and strip whitespace before comparing.
    na_equal       : Treat NaN as a comparable value (filled with a sentinel).
    expand         : If True, return one row per missing occurrence instead of
                     an aggregated report (new_count / ref_count / diff_count).
    fix_typo       : Silently rename 'filemane' → 'filename' if found.
    """
    w = new_df.copy()
    r = ref_df.copy()

    if fix_typo:
        for df in (w, r):
            if 'filemane' in df.columns and 'filename' not in df.columns:
                df.rename(columns={'filemane': 'filename'}, inplace=True)

    if on is None:
        keys = list(w.columns.intersection(r.columns))
        if not keys:
            raise KeyError('No common columns between new_df and ref_df.')
    else:
        keys = list(on)
        for side, df in [('new_df', w), ('ref_df', r)]:
            miss = [k for k in keys if k not in df.columns]
            if miss:
                raise KeyError(f'Missing key columns in {side}: {miss}')

    w_k = w[keys].copy()
    r_k = r[keys].copy()

    if normalize:
        for col in keys:
            w_k[col] = w_k[col].astype('string').str.strip()
            r_k[col] = r_k[col].astype('string').str.strip()

    sentinel = '____NA_SENTINEL____'
    if na_equal:
        w_k = w_k.fillna(sentinel)
        r_k = r_k.fillna(sentinel)

    new_counts = (w_k.groupby(keys, dropna=False, sort=False)
                     .size().reset_index(name='new_count'))
    ref_counts = (r_k.groupby(keys, dropna=False, sort=False)
                     .size().reset_index(name='ref_count'))

    merged = new_counts.merge(ref_counts, on=keys, how='left')
    merged['ref_count']  = merged['ref_count'].fillna(0).astype(int)
    merged['diff_count'] = merged['new_count'] - merged['ref_count']

    result = merged[merged['diff_count'] > 0].reset_index(drop=True)
    if not expand:
        return result.sort_values('diff_count', ascending=False).reset_index(drop=True)
    return result.loc[
        result.index.repeat(result['diff_count']), keys
    ].reset_index(drop=True)


def anti_join(
    new_df: pd.DataFrame,
    ref_df: pd.DataFrame,
    keys=('filename', 'station', 'location'),
    normalize: bool = True,
    na_equal: bool = False,
) -> pd.DataFrame:
    """Return rows of new_df that have no match in ref_df on the given keys."""
    w = new_df.copy()
    r = ref_df.copy()

    for df in (w, r):
        if 'filemane' in df.columns and 'filename' not in df.columns:
            df.rename(columns={'filemane': 'filename'}, inplace=True)

    for side, df in [('new_df', w), ('ref_df', r)]:
        miss = [k for k in keys if k not in df.columns]
        if miss:
            raise KeyError(f'Missing keys in {side}: {miss}')

    if normalize:
        for df in (w, r):
            for k in keys:
                df[k] = df[k].astype('string').str.strip()

    sentinel = '__NA__'
    if na_equal:
        for df in (w, r):
            for k in keys:
                df[k] = df[k].fillna(sentinel)

    ref = r[list(keys)].drop_duplicates()
    return (
        w.merge(ref, how='left', on=list(keys), indicator=True)
         .query('_merge == "left_only"')
         .drop(columns='_merge')
    )


print('✅ Cross-validation functions defined: count_missing(), anti_join()')

In [ ]:
# ── Cross-validation example ──────────────────────────────────────────────────
# Compare the current extraction against a previous reference run.
# Uncomment and adjust the path as needed.

# ref_df = pd.read_csv('waveform_summary_reference.csv',
#                      dtype={'station': str, 'location': str})
#
# --- Multiset comparison ---
# report = count_missing(df_all, ref_df, normalize=True, na_equal=True)
# print(f'Unique key combinations in new but not in reference : {len(report)}')
# print(f'Total extra rows (sum of diff_count)               : {report["diff_count"].sum()}')
# display(report.head(20))
#
# --- Anti-join ---
# unmatched = anti_join(df_all, ref_df)
# print(f'Rows with no match in reference: {len(unmatched)}')
# display(unmatched.head())
#
# --- Filename set comparison ---
# fn_new = set(df_all['filename'].unique())
# fn_ref = set(ref_df['filename'].unique())
# print(f'Only in new : {len(fn_new - fn_ref)}')
# print(f'Only in ref : {len(fn_ref - fn_new)}')